# Notebook 04 — Stage 1A: canonical schema and presence flags

**Role in the pipeline:** first transformation stage on the canonical
data critical path. Reads the QC-corrected `derived.csv` produced by
Notebook 02 and produces two analysis-ready frames plus an audit trail.

```text
02_ta_ph_qc.ipynb
   └── sheet_<x>/data/derived.csv
                     │
                     ▼
          04_stage1a.ipynb         ◄── THIS NOTEBOOK
            • resolve canonical column names (alias resolution)
            • normalise TA units and pH scale labels
            • flag out-of-range values (advisory, never destructive)
            • flag possible duplicates
            • write staged.csv + analysis_ready.csv (+ Parquet)
                              │
                              ▼
                  05_stage1b.ipynb  →  06_stage2  →  ...
```

**What "Stage 1A" produces**

- `staged.csv` — every column the input had, *plus* the canonical columns
  copied from their aliases, *plus* the flag columns. Use this for
  debugging the alias resolution.
- `analysis_ready.csv` — same rows, but columns reordered into the
  canonical export order. **This is what Stage 1B reads.**
- Audit logs in `logs/`: rename audit, canonical inventory,
  missingness inventory, effective config, and `manifest.json`.
- A one-page `report.md`.

See `04_stage1a.README.md` for the design rationale.


## Parameters

Single tagged `parameters` cell. The default `INPUT_CSV` points at the
*new* short path Notebook 02 writes
(`sheet_<x>/data/derived.csv`), not the old `<stem>__0__derived.csv`.


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# Default: the canonical Notebook 02 output for sheet 0. Adjust SHEET in
# the path if you need another sheet, or override OUT_DIR for an alternate
# destination.
INPUT_CSV = r"C:\Users\OA_2023-03\OneDrive\Habitat Suitabilty model\OA\data_1\oa_prelim_data__qc_outputs\sheet_0\data\derived.csv"
OUT_DIR = r"C:\Users\OA_2023-03\OneDrive\Habitat Suitabilty model\OA\data_1\oa_stage1a_outputs"

# --- Config override (optional) -------------------------------------
# Path to a .json/.yml/.yaml file that deep-merges onto DEFAULT_CONFIG in
# oa_schema.py. Use this to override range bounds, add new aliases for a
# column, change the canonical export order, etc., WITHOUT editing code.
CONFIG_PATH = None

# --- Stage 1A behaviour --------------------------------------------
# Comma-separated override list for the duplicate-key tuple, e.g.
# "sample_id,replicate_id,sample_date,station_id". None -> auto-pick from
# config["duplicate_key_candidates"] (first set where all columns exist
# AND have at least one non-NA value -- see oa_schema.choose_duplicate_keys
# for the rationale).
DUPLICATE_KEYS = None

# If True (default), copy resolved aliases into canonical columns AND
# keep the original column too. Useful while iterating. Set to False for a
# leaner output where the alias is renamed in place.
PRESERVE_ORIGINAL_COLUMNS = True

# If True, skip Parquet writes (CSV only). Parquet requires pyarrow or
# fastparquet; a failed Parquet write is logged in the manifest but does
# NOT fail the notebook.
NO_PARQUET = False

# Useful during development: prepare everything but write nothing.
DRY_RUN = False

# If True, print large tables in console form. If False, use rich
# display() (better in Jupyter).
PRINT_COLUMNS = False


## Setup

No `%pip install` — `pandas` is required; `pyarrow`/`fastparquet`
(for Parquet) and `pyyaml` (for YAML configs) are *optional*. Missing
them yields a clean fallback, not a crash.

The QC math, the canonical schema, and the range policy each live in
their own module (`oa_qc_ta_ph.py`, `oa_schema.py`, `oa_policy.py`). The
notebook itself is just the orchestrator.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from oa_pipeline.common import (
    build_flag_summary,
    coerce_datetime,
    coerce_numeric,
    die,
    ensure_dir,
    make_missingness_table,
    md_table_from_df,
    utc_stamp,
    write_csv_and_parquet,
    write_json,
    write_text,
)
from oa_pipeline.policy import (
    add_stage_range_flags,
    policy_from_config,
)
from oa_pipeline.schema import (
    add_canonical_presence_flags,
    add_duplicate_flags,
    apply_canonical_schema,
    build_canonical_export,
    build_canonical_inventory,
    build_numeric_candidates_for_canonical,
    choose_duplicate_keys,
    load_config,
)


## Load input CSV and config

In [ ]:
src_csv = Path(INPUT_CSV).expanduser().resolve()
if not src_csv.exists():
    die(f"File not found: {src_csv}\n"
        f"Did Notebook 02 run successfully? Stage 1A reads its 'derived.csv'.")
if src_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {src_csv.name}")

out_root = ensure_dir(Path(OUT_DIR).expanduser().resolve())

config, config_source = load_config(CONFIG_PATH)
range_policy = policy_from_config(config)

df = pd.read_csv(src_csv)
if df.empty:
    die(f"Input CSV is empty: {src_csv}")

df.columns = [str(c).strip() for c in df.columns]
df["source_file"] = str(src_csv)
df["stage1a_processed_utc"] = utc_stamp()

# Cast sample_date if present (alias-resolved below, but cast here in case
# the source already uses the canonical name).
if "sample_date" in df.columns:
    coerce_datetime(df, "sample_date")

print(f"Source CSV    : {src_csv}")
print(f"Rows          : {len(df):,}")
print(f"Columns       : {df.shape[1]}")
print(f"Config source : {config_source if config_source else 'DEFAULT_CONFIG'}")


## Apply canonical schema

For each canonical column (e.g. `salinity`, `ta_umol_kg`,
`latitude_deg`), find the first matching alias in the source frame and
copy / rename it. Unit strings (`umol kg-1`) and pH-scale labels
(`total` / `free` / `seawater` / `nbs`) are normalised. Default
provenance columns (`carbonate_solver = "PyCO2SYS"`, etc.) are filled
where empty.

This step never drops rows. It is purely additive plus rename / copy.


In [ ]:
df, rename_audit, source_lookup = apply_canonical_schema(
    df,
    config=config,
    preserve_original_columns=PRESERVE_ORIGINAL_COLUMNS,
)

# Numeric coercion (string-looking numerics get parsed; bad values -> NaN).
coerce_numeric(df, build_numeric_candidates_for_canonical())

# sample_date may have been alias-resolved into existence; coerce now.
if "sample_date" in df.columns:
    coerce_datetime(df, "sample_date")

print(f"Canonical actions applied : {len(rename_audit)}")
print(f"Total columns after schema: {df.shape[1]}")


## Add flag columns

Three classes of flag, all advisory (a flag never deletes a row):

- **Range flags** (`flag_*_out_of_range`) for `salinity`, `ta_umol_kg`,
  `ph_observed`, `ph_calculated`, `depth_m`, `latitude_deg`,
  `longitude_deg`. Bounds come from the `range_policy` block of the
  config.
- **Presence flags**: `flag_required_core_missing`, `flag_ta_units_*`,
  `flag_ph_scale_observed_missing`, `flag_pressure_output_dbar_missing`.
- **Duplicate flag** (`flag_possible_duplicate`): chooses a key tuple
  from `config["duplicate_key_candidates"]` -- the first set where all
  columns exist *and* have at least one non-NA value (the latter check
  is the one bug-fix to the original Stage 1A logic, see README §6).


In [ ]:
# Range + presence
add_stage_range_flags(df, policy=range_policy)
add_canonical_presence_flags(df, config=config)

# Duplicates
dup_override = None
if DUPLICATE_KEYS:
    dup_override = [x.strip() for x in str(DUPLICATE_KEYS).split(",") if x.strip()]

dup_keys = choose_duplicate_keys(df, config, override_keys=dup_override)
n_dup = add_duplicate_flags(df, dup_keys)

print(f"Duplicate keys used      : {dup_keys}")
print(f"Rows flagged as duplicate: {n_dup}")


## Build analysis-ready frame and inventories

`build_canonical_export` reorders columns into the canonical export
order from the config; columns not in that list go to the right-hand end
(so nothing is lost). The result is `analysis_ready_df` — what Stage 1B
will read.

`build_canonical_inventory` and `make_missingness_table` are summary
tables for the audit log.


In [ ]:
analysis_ready_df = build_canonical_export(df, config=config)
canonical_inventory = build_canonical_inventory(df, config=config)
missing_tbl = make_missingness_table(df)

print(f"Canonical columns present: "
      f"{int(canonical_inventory['present'].sum())} / {len(canonical_inventory)}")


## Quick previews (interactive run only)

In [ ]:
if PRINT_COLUMNS:
    print("=== Missingness (top 15) ===")
    print(missing_tbl.head(15).to_string(index=False))
    print("\n=== Rename audit ===")
    print(rename_audit.to_string(index=False))
    print("\n=== Canonical inventory (first 20) ===")
    print(canonical_inventory.head(20).to_string(index=False))
elif display is not None:
    display(missing_tbl.head(15))
    display(rename_audit)
    display(canonical_inventory.head(20))
    display(analysis_ready_df.head(10))


## Prepare output paths

Layout — short filenames, role in the parent folder, no workbook stem
infix, no `__stage1a__` accumulation:

```
<OUT_DIR>/
    data/
        staged.csv                (and .parquet)
        analysis_ready.csv        (and .parquet)   ◄── Stage 1B input
    reports/
        report.md
    logs/
        manifest.json
        effective_config.json
        rename_audit.csv
        canonical_inventory.csv
        missingness.csv
```

The original notebook nested everything under
`oa_stage1a_outputs/<input-stem>/...` which both produced very long paths
and accumulated stage tags. With Papermill, you can run Stage 1A on a
different `INPUT_CSV` and just point `OUT_DIR` somewhere new — separate
directory per run, no filename juggling. (Same principle as the JWST
pipeline recommendation: "use `output_dir` to place the results in a
different directory instead of using `output_file` to rename".)


In [ ]:
data_dir    = ensure_dir(out_root / "data")
reports_dir = ensure_dir(out_root / "reports")
logs_dir    = ensure_dir(out_root / "logs")

paths = {
    "staged_csv":              data_dir    / "staged.csv",
    "staged_parquet":          data_dir    / "staged.parquet",
    "analysis_ready_csv":      data_dir    / "analysis_ready.csv",
    "analysis_ready_parquet":  data_dir    / "analysis_ready.parquet",
    "report_md":               reports_dir / "report.md",
    "manifest_json":           logs_dir    / "manifest.json",
    "effective_config_json":   logs_dir    / "effective_config.json",
    "rename_audit_csv":        logs_dir    / "rename_audit.csv",
    "canonical_inventory_csv": logs_dir    / "canonical_inventory.csv",
    "missingness_csv":         logs_dir    / "missingness.csv",
}
print(f"Output root: {out_root}")


## Write outputs

`DRY_RUN = True` skips the writes (useful while you iterate on a config).
Parquet failures are logged in the manifest but do not fail the notebook.


In [ ]:
parquet_written = {"staged": False, "analysis_ready": False}
parquet_errors  = {"staged": None,  "analysis_ready": None}

if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    if NO_PARQUET:
        df.to_csv(paths["staged_csv"], index=False)
        analysis_ready_df.to_csv(paths["analysis_ready_csv"], index=False)
        parquet_errors["staged"] = "Parquet disabled by user"
        parquet_errors["analysis_ready"] = "Parquet disabled by user"
    else:
        ok_s, err_s = write_csv_and_parquet(df, paths["staged_csv"], paths["staged_parquet"])
        parquet_written["staged"], parquet_errors["staged"] = ok_s, err_s
        ok_a, err_a = write_csv_and_parquet(analysis_ready_df, paths["analysis_ready_csv"], paths["analysis_ready_parquet"])
        parquet_written["analysis_ready"], parquet_errors["analysis_ready"] = ok_a, err_a

    rename_audit.to_csv(paths["rename_audit_csv"], index=False)
    canonical_inventory.to_csv(paths["canonical_inventory_csv"], index=False)
    missing_tbl.to_csv(paths["missingness_csv"], index=False)
    write_json(paths["effective_config_json"], config)

    # Markdown report
    flag_summary = build_flag_summary(df)
    rename_section = md_table_from_df(rename_audit, max_rows=200) if not rename_audit.empty else "_(No canonical actions applied)_"
    inventory_section = md_table_from_df(canonical_inventory, max_rows=200) if not canonical_inventory.empty else "_(No canonical inventory available)_"
    flag_section = md_table_from_df(flag_summary, max_rows=200) if not flag_summary.empty else "_(No flags were added.)_"

    report_md_text = f"""# Stage 1A Report

**Generated:** {utc_stamp()}
**Source CSV:** `{src_csv}`
**Config source:** `{config_source if config_source else "DEFAULT_CONFIG"}`
**Rows:** {len(df):,}
**Columns:** {df.shape[1]:,}
**Preserve original columns:** `{PRESERVE_ORIGINAL_COLUMNS}`

## What this script does
1. Resolve canonical alias columns from the source frame
2. Normalise TA-unit and pH-scale labels to canonical strings
3. Add range, presence and duplicate flags (advisory; never destructive)
4. Write a staged frame (everything) and an analysis-ready frame
   (canonical export order)

## Canonical column actions
{rename_section}

## Canonical inventory
{inventory_section}

## Missingness inventory (top 50)
{md_table_from_df(missing_tbl.head(50), max_rows=200)}

## Duplicate checks
- Duplicate keys used: `{dup_keys}`
- Rows flagged as possible duplicates: **{n_dup}**

## Range policy used
- Salinity: {range_policy.sal_min} to {range_policy.sal_max}
- TA: {range_policy.ta_min} to {range_policy.ta_max} umol/kg
- pH: {range_policy.ph_min} to {range_policy.ph_max}
- Depth: {range_policy.depth_min} to {range_policy.depth_max} m
- Latitude: {range_policy.lat_min} to {range_policy.lat_max}
- Longitude: {range_policy.lon_min} to {range_policy.lon_max}

## Flag summary
{flag_section}

## Main outputs (paths)
- staged CSV         : `{paths["staged_csv"]}`
- analysis-ready CSV : `{paths["analysis_ready_csv"]}`  (Stage 1B reads this)
"""
    write_text(paths["report_md"], report_md_text)

    # Manifest -- single source of truth for what just happened
    manifest = {
        "notebook": "04_stage1a",
        "generated_utc": utc_stamp(),
        "input_csv": str(src_csv),
        "output_root": str(out_root),
        "config_source": config_source,
        "parameters": {
            "INPUT_CSV": str(src_csv),
            "OUT_DIR": str(out_root),
            "CONFIG_PATH": CONFIG_PATH,
            "DUPLICATE_KEYS": DUPLICATE_KEYS,
            "PRESERVE_ORIGINAL_COLUMNS": PRESERVE_ORIGINAL_COLUMNS,
            "NO_PARQUET": NO_PARQUET,
            "DRY_RUN": DRY_RUN,
            "PRINT_COLUMNS": PRINT_COLUMNS,
        },
        "policy": {
            "range_policy": asdict(range_policy),
            "duplicate_keys_used": dup_keys,
            "required_columns": config.get("required_columns", []),
            "canonical_export_order": config.get("canonical_export_order", []),
        },
        "row_counts": {
            "staged_rows": int(len(df)),
            "analysis_ready_rows": int(len(analysis_ready_df)),
            "possible_duplicate_rows": int(n_dup),
        },
        "canonical_summary": {
            "n_present": int(canonical_inventory["present"].sum()),
            "n_missing": int((~canonical_inventory["present"]).sum()),
        },
        "parquet_written": parquet_written,
        "parquet_errors": parquet_errors,
        "outputs": {k: str(v) for k, v in paths.items()},
        "package_versions": {
            "python": sys.version.split()[0],
            "pandas": pd.__version__,
        },
    }
    write_json(paths["manifest_json"], manifest)

    print("\nStage 1A complete.")
    print(f"  -> Stage 1B input: {paths['analysis_ready_csv']}")


## Review outputs

In [ ]:
if not DRY_RUN:
    outputs_df = pd.DataFrame(
        {"output_name": list(paths.keys()), "path": [str(p) for p in paths.values()]}
    )
    if display is not None:
        display(outputs_df)
        print("\nRename audit:")
        display(rename_audit.head(20))
        print("\nCanonical inventory (first 20):")
        display(canonical_inventory.head(20))
        print("\nAnalysis-ready preview:")
        display(analysis_ready_df.head(10))
    else:
        print(outputs_df.to_string(index=False))
